In [5]:
# -*- coding: utf-8 -*-
# Author: Qinghua Liu <liu.11085@osu.edu>
# License: Apache-2.0 License

import pandas as pd
import numpy as np
import torch
import random, argparse, time, os, logging
from TSB_AD.evaluation.metrics import get_metrics
from TSB_AD.utils.slidingWindows import find_length_rank
from TSB_AD.model_wrapper import *
from TSB_AD.HP_list import Optimal_Multi_algo_HP_dict

# seeding
seed = 2024
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

print("CUDA available: ", torch.cuda.is_available())
print("cuDNN version: ", torch.backends.cudnn.version())

# List of univariable models. Removed AnomalyTransformer due to its CUDA requirement.
# Removed Donut due to its internal error.
# Also not included: all transformer model, as well as licensed models (NORMA, Series2Graph)
model_list = ['IForest', 'LOF', 'PCA', 'HBOS', 'OCSVM', 'MCD', 'KNN', 'KMeansAD', 'KShapeAD',
              'COPOD', 'CBLOF', 'EIF', 'RobustPCA', 'AutoEncoder', 'CNN', 'LSTMAD', 'TranAD', 
              'OmniAnomaly', 'USAD', 'FITS']

for model in model_list:
    try:
        ## ArgumentParser
        parser = argparse.ArgumentParser(description='Generating Anomaly Score')
        parser.add_argument('--dataset_dir', type=str, default='/Users/kai/Documents/Time_Series_Anomaly_Detection_Seminar/Time-Series-Anomaly-Detection-Seminar/TSB-AD/Datasets/TSB-AD-M')
        parser.add_argument('--file_list', type=str, default='/Users/kai/Documents/Time_Series_Anomaly_Detection_Seminar/Time-Series-Anomaly-Detection-Seminar/TSB-AD/Datasets/File_List/one_set_test_multi.csv')
        parser.add_argument('--score_dir', type=str, default='eval/score/multi/')
        parser.add_argument('--save_dir', type=str, default='eval/metrics/multi/')
        parser.add_argument('--no-save', action='store_false', dest='save', default=True, help='Disable saving')
        parser.add_argument('--AD_Name', type=str, default=model)

        args = parser.parse_args([])

        os.makedirs(args.score_dir, exist_ok=True)
        os.makedirs(args.save_dir, exist_ok=True)

        target_dir = os.path.join(args.score_dir, args.AD_Name)
        target_dir_metrics = os.path.join(args.save_dir, args.AD_Name)
        os.makedirs(target_dir, exist_ok = True)
        os.makedirs(target_dir_metrics, exist_ok = True)
        logging.basicConfig(filename=f'{target_dir}/000_run_{args.AD_Name}.log', level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

        file_list = pd.read_csv(args.file_list)['file_name'].values
        Optimal_Det_HP = Optimal_Multi_algo_HP_dict[args.AD_Name]
        print('Optimal_Det_HP: ', Optimal_Det_HP)

        write_csv = []
        for filename in file_list:
            if os.path.exists(target_dir+'/'+filename.split('.')[0]+'.npy'): continue
            print('Processing:{} by {}'.format(filename, args.AD_Name))

            file_path = os.path.join(args.dataset_dir, filename)
            df = pd.read_csv(file_path).dropna()
            data = df.iloc[:, 0:-1].values.astype(float)
            label = df['Label'].astype(int).to_numpy()
            # print('data: ', data.shape)
            # print('label: ', label.shape)

            feats = data.shape[1]
            slidingWindow = find_length_rank(data[:,0].reshape(-1, 1), rank=1)
            train_index = filename.split('.')[0].split('_')[-3]
            data_train = data[:int(train_index), :]

            start_time = time.time()

            if args.AD_Name in Semisupervise_AD_Pool:
                output = run_Semisupervise_AD(args.AD_Name, data_train, data, **Optimal_Det_HP)
            elif args.AD_Name in Unsupervise_AD_Pool:
                output = run_Unsupervise_AD(args.AD_Name, data, **Optimal_Det_HP)
            else:
                raise Exception(f"{args.AD_Name} is not defined")

            end_time = time.time()
            run_time = end_time - start_time

            if isinstance(output, np.ndarray):
                logging.info(f'Success at {filename} using {args.AD_Name} | Time cost: {run_time:.3f}s at length {len(label)}')
                np.save(f"{target_dir}/{args.AD_Name}_{filename.split('.')[0]}.npy", output)
            else:
                logging.error(f'At {filename}: '+output)

            ### whether to save the evaluation result
            if args.save:
                print("args.save is triggering correctly")
                try:
                    evaluation_result = get_metrics(output, label, slidingWindow=slidingWindow)
                    print('evaluation_result: ', evaluation_result)
                    list_w = list(evaluation_result.values())
                except Exception as e:
                    logging.error(f"Error calling get_metrics for {filename}: {e}")
                    logging.error(f"Output shape: {output.shape}, Label shape: {label.shape}, Sliding window: {slidingWindow}")
                    # Optionally log parts of the arrays if helpful, e.g.:
                    # logging.error(f"Output sample: {output[:10]}")
                    # logging.error(f"Label sample: {label[:10]}")
                    list_w = [0]*9
                list_w.insert(0, run_time)
                list_w.insert(0, filename)
                write_csv.append(list_w)

                ## Temp Save
                col_w = list(evaluation_result.keys())
                col_w.insert(0, 'Time')
                col_w.insert(0, 'file')
                w_csv = pd.DataFrame(write_csv, columns=col_w)
                w_csv.to_csv(f"{target_dir_metrics}/{args.AD_Name}.csv", index=False)
    except Exception as e:
        print(f"{model}_not working. Error: {e}")

CUDA available:  False
cuDNN version:  None
Optimal_Det_HP:  {'n_estimators': 25, 'max_features': 0.8}
Processing:057_SMD_id_1_Facility_tr_4529_1st_4629.csv by IForest
args.save is triggering correctly


/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


evaluation_result:  {'AUC-PR': 0.10249559890981996, 'AUC-ROC': 0.7964200011730593, 'VUS-PR': 0.10219084641197465, 'VUS-ROC': 0.8138758836724986, 'Standard-F1': 0.16677488989021713, 'PA-F1': 0.5163297045101088, 'Event-based-F1': 0.22417582417582368, 'R-based-F1': 0.14918943544982322, 'Affiliation-F': 0.8094138452538187}
Optimal_Det_HP:  {'n_neighbors': 50, 'metric': 'euclidean'}
Processing:057_SMD_id_1_Facility_tr_4529_1st_4629.csv by LOF
args.save is triggering correctly


/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


evaluation_result:  {'AUC-PR': 0.04135624987731625, 'AUC-ROC': 0.6331650354340448, 'VUS-PR': 0.05222054862520202, 'VUS-ROC': 0.7031771831345714, 'Standard-F1': 0.08761717113210778, 'PA-F1': 0.6881405563689604, 'Event-based-F1': 0.13216280925778118, 'R-based-F1': 0.10685608085203316, 'Affiliation-F': 0.7506056620292166}
Optimal_Det_HP:  {'n_components': 0.25}
Processing:057_SMD_id_1_Facility_tr_4529_1st_4629.csv by PCA
args.save is triggering correctly


...

evaluation_result:  {'AUC-PR': 0.5233450407172351, 'AUC-ROC': 0.977780365981787, 'VUS-PR': 0.4743586167651548, 'VUS-ROC': 0.953574347346165, 'Standard-F1': 0.5072227465832032, 'PA-F1': 0.6326102169349195, 'Event-based-F1': 0.5273377618804288, 'R-based-F1': 0.5878465462241427, 'Affiliation-F': 0.9150422834133983}
Optimal_Det_HP:  {'win_size': 100, 'lr': 0.001}
Processing:057_SMD_id_1_Facility_tr_4529_1st_4629.csv by FITS
----- GPU is unavailable -----
----- Using CPU -----


Testing: : 100%|██████████| 185/185 [00:02<00:00, 81.13it/s]


args.save is triggering correctly


/opt/homebrew/Caskroom/miniconda/base/envs/TSB-AD/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


evaluation_result:  {'AUC-PR': 0.07831644073912916, 'AUC-ROC': 0.8220071205981582, 'VUS-PR': 0.09218956374621243, 'VUS-ROC': 0.8479138368122967, 'Standard-F1': 0.14331956806288879, 'PA-F1': 0.8007279344858963, 'Event-based-F1': 0.3199999999999995, 'R-based-F1': 0.14375591605199453, 'Affiliation-F': 0.8104132699343095}
